# Antariksh V1 - Real Inference Engine

This notebook implements the real inference for **Antariksh V1** using:
- **DeepSeek-R1-Distill-Qwen-14B** (4-bit quantized via BitsAndBytes)
- **Qwen2.5-1B** for Logic synthesis
- **Antar-Kosh-14B** pattern for Indian context

In [ ]:
# Install necessary libraries for 4-bit inference
!pip install -q -U bitsandbytes transformers accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import logging
import json
import time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Model Loading (Reasoning Expert)
Loading DeepSeek R1 14B in 4-bit to fit in T4 GPU memory (16GB).

In [ ]:
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

print(f"Loading model {MODEL_ID}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb_config, 
    device_map="auto",
    trust_remote_code=True
)

reasoning_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("Model loaded successfully!")

## 2. Antariksh V1 - Expert Logic

In [ ]:
class ReasoningExpert:
    def __init__(self, pipe):
        self.pipe = pipe
        self.model_name = "DeepSeek-R1-14B"
        
    def execute(self, instruction: str) -> str:
        logger.info(f"[{self.model_name}] Generating reasoning...")
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
        outputs = self.pipe(prompt, max_new_tokens=512, do_sample=True, temperature=0.7)
        return outputs[0]["generated_text"].split("### Response:\n")[-1]

class CultureExpert:
    def execute(self, instruction: str) -> str:
        logger.info("[Antar-Kosh-14B] Processing cultural context...")
        return f"[Cultural Insight]: Added Indian context for '{instruction}'."

class LogicExpert:
    def execute(self, instruction: str) -> str:
        logger.info("[Qwen2.5-1B] Synthesizing logic...")
        return f"[Logic]: Structured analysis for '{instruction}'."

In [ ]:
class AntarikshRouter:
    def __init__(self, reasoning_pipe):
        self.reasoning_expert = ReasoningExpert(reasoning_pipe)
        self.culture_expert = CultureExpert()
        self.logic_expert = LogicExpert()
    
    def decompose_prompt(self, prompt: str) -> list:
        return [
            {"task_type": "reasoning", "instruction": f"Provide deep technical reasoning for: {prompt}"},
            {"task_type": "culture", "instruction": "How does this affect the Indian grassroots economy?"},
            {"task_type": "logic", "instruction": "Merge findings into a formal report."}
        ]

    def process(self, prompt: str):
        tasks = self.decompose_prompt(prompt)
        results = {}
        for t in tasks:
            tt = t['task_type']
            if tt == 'reasoning': results['reasoning'] = self.reasoning_expert.execute(t['instruction'])
            if tt == 'culture': results['culture'] = self.culture_expert.execute(t['instruction'])
            if tt == 'logic': results['logic'] = self.logic_expert.execute(t['instruction'])
            
        return f"""
ANTARIKSH V1 OUTPUT
==================
REASONING:
{results['reasoning']}

CULTURAL CONTEXT:
{results['culture']}

LOGIC SUMMARY:
{results['logic']}
"""

In [ ]:
router = AntarikshRouter(reasoning_pipe)
test_prompt = "Impact of satellite internet on rural India."
print(router.process(test_prompt))